
# Data Retrieval Guide — Life Criticality & Budget Sources

---

## Access Tiers at a Glance

| Tier                | Sources                                                                                                 |
|---------------------|--------------------------------------------------------------------------------------------------------|
| **API / no auth**   | WHO GHO OData, Nikshay (POST scrape), WHO GHED                                                         |
| **Free registration** | IHME GBD + ICMR state data (one GHDx account), NFHS (dhsprogram.com), GeM (data.gov.in)              |
| **Login / empanelment** | PM-JAY HBP (NHA SETU), GeM API                                                                     |
| **PDF extraction**  | SRS, NCRB, MoRTH, CGHS rates, IPHS 2022, CPWD SOR, NHA reports                                         |
| **Web scrape**      | CPPP/tender portals                                                                                    |

---

## Life Criticality Sources

**1. WHO GHO OData API**  
- **URL:** [https://ghoapi.azureedge.net/api/](https://ghoapi.azureedge.net/api/)  
- **Auth:** None  
- **Format:** JSON (OData)  
- **Key indicators:**  
  - `GHE_DALYNUM` / `GHE_DALYRATE` — DALYs by cause, sex, age  
  - `GHE_YLLNUM` — Years of Life Lost  
  - `MORT_100` — child deaths by cause (works at country level: `?$filter=SpatialDim eq 'IND'`)  
- **Limitation:** Cause-of-death GHE is regional aggregates only — `SpatialDim eq 'IND'` works for mortality indicators but not full GHE cause breakdown  
- **India state-level:** No — country (IND) only  

**2. IHME / GBD Results + ICMR State-Level (one account, two datasets)**  
- **URL:** [GBD Results Tool](https://vizhub.healthdata.org/gbd-results/) + [GHDx](https://ghdx.healthdata.org/)  
- **Auth:** Single free account at [GHDx](https://ghdx.healthdata.org/) — unlocks both  
- **Format:** CSV zip export  
- **Key columns:** `measure_name`, `location_name`, `cause_name`, `metric_name`, `year`, `val`, `upper`, `lower`, `sex_name`, `age_name`  
- **India state-level:** Yes — all Indian states, DALYs/YLLs/deaths by cause, 1990–2021  
- **ICMR data:** Same GHDx login — the ICMR/IHME "India: Health of the Nation's States" dataset is hosted there; supplementary Excel is also attached to the Lancet paper  

**3. NFHS (National Family Health Survey)**  
- **URL:** [NFHS 2020](https://dhsprogram.com/data/dataset/India_Standard-DHS_2020.cfm)  
- **Auth:** Free registration at dhsprogram.com (1–3 day approval, institutional email preferred)  
- **Format:** Stata (.dta) / SPSS (.sav) per survey module (women's, household, children's, facility)  
- **Key columns:** `state`, `district`, `rural_urban`, `anc_coverage`, `institutional_delivery_pct`, `neonatal_mortality_rate`, `immunization_pct`, `mmr`  
- **India state-level:** Yes, down to district; no bulk CSV for district fact sheets  

**4. SRS (Sample Registration System)**  
- **URL:** [SRS Search](https://censusindia.gov.in/nada/index.php/catalog?sk=SRS)  
- **Auth:** None — PDFs are public  
- **Format:** PDF → extract with pdfplumber (tables are text-based, not scanned)  
- **Key data:** State-level MMR, infant mortality, cause of death distribution, rural/urban  
- **Limitation:** State level only, no district; no machine-readable bulk download  

**5. HMIS**  
- **URL:** [HMIS Portal](https://hmis.nhp.gov.in/) (Excel export via portal filters) or [data.gov.in](https://data.gov.in) (search "HMIS", free registration for API key)  
- **Auth:** None on portal; free registration on data.gov.in  
- **Format:** Excel (portal) / CSV (data.gov.in)  
- **Key columns:** `state`, `district`, `block`, `facility_type`, `institutional_deliveries`, `anc_registrations`, `immunizations`, `month`, `year`  
- **Note:** Portal is intermittently inaccessible from non-Indian IPs — data.gov.in is the more reliable programmatic path  

**6. NCRB ADSI (Accidental Deaths & Suicides)**  
- **URL:** [NCRB ADSI](https://www.ncrb.gov.in/accidental-deaths-suicides-in-india-adsi.html)  
- **Auth:** None  
- **Format:** PDF → pdfplumber (consistent table structure across years)  
- **Key data:** State/city accidental deaths by cause (road, drowning, fire, disaster), age, sex  

**7. MoRTH Road Accident Reports**  
- **URL:** [MoRTH Reports](https://morth.nic.in/road-accident-in-india) (JavaScript SPA — navigate to year then download)  
- **Auth:** None  
- **Format:** PDF → pdfplumber  
- **Key data:** State/district road accidents, fatalities, severity, vehicle type — more granular than NCRB for trauma  

**8. Nikshay (TB)**  
- **Endpoint:** POST `https://reports.nikshay.in/TBNotification/GetTBNotificationTab`  
- **Auth:** None — public POST endpoint  
- **Payload:** `Year=2024&submitType=1&submitHierarchy=All`  
- **Format:** HTML table in response → parse with BeautifulSoup → CSV  
- **Key columns:** `state`, `district`, `tb_notifications_public`, `tb_notifications_private`, `treatment_success_rate`, `year`  
- **Year range:** 2015–2026  

**9. ICMR-NINE / NCRP (Cancer)**  
- **URL:** [ICMR-NINE Reports](https://icmrnine.org/All_Reports.aspx)  
- **Auth:** None  
- **Format:** PDF → pdfplumber (28 population-based cancer registries)  
- **Key data:** Incidence by cancer site (breast, cervix, oral, lung, colorectal), age-standardized rates, registry (~state) level  

---

## Budget Sources

**10. PM-JAY Health Benefit Package Rates**  
- **URL:** [NHA SETU](https://setu.pmjay.gov.in) (empanelment login) or state SHA mirrors  
- **Auth:** NHA SETU registration; older HBP versions freely mirrored  
- **Format:** Excel (.xlsx), ~2,000–3,000 rows  
- **Key columns:** `package_code`, `package_name`, `specialty`, `sub_specialty`, `rate_INR`, `implant_limit`, `daycare_flag`  

**11. CGHS Package Rates**  
- **URL:** [CGHS](https://cghs.nic.in/) (NIC mirror; cghs.gov.in is persistently slow externally)  
- **Auth:** None for PDFs  
- **Format:** PDF (Office Memorandum circulars) → extract; unofficial Excel conversions exist on state health dept sites  
- **Key columns:** `procedure_name`, `NABH_rate`, `non_NABH_rate`, `specialty`, `package_code`  

**12. GeM Equipment Prices**  
- **URL:** [GeM Orders](https://data.gov.in/resource/gem-orders) (quarterly transaction data) or [GeM Portal](https://mkp.gem.gov.in/) (scrape)  
- **Auth:** Free registration on data.gov.in for CSV download; GeM API requires buyer/seller registration  
- **Format:** JSON (API) / CSV (data.gov.in) / HTML scrape  
- **Key columns:** `product_name`, `category`, `L1_price_INR`, `brand`, `specifications`, `bid_date`  

**13. CPPP / State Tender Portals (Awarded Prices)**  
- **URL:** [GeM All Bids](https://bidplus.gem.gov.in/all-bids) (best for medical equipment, structured) or [CPPP Tender Search](https://eprocure.gov.in/cppp/tendersearch) (POST scrape)  
- **Auth:** None for web scraping  
- **Format:** HTML scrape only — no bulk API  
- **Key columns:** `item_description`, `awarded_rate_INR`, `vendor`, `award_date`, `ministry_state`  

**14. IPHS 2022 Norms (confirmed public URLs)**  
- **Auth:** None  
- **Format:** PDF → PyMuPDF (fitz) — text-based tables, not scanned  

| Volume | Facility              | URL                                                                                      |
|--------|-----------------------|-----------------------------------------------------------------------------------------|
| Vol 1  | SDH + District Hosp.  | [Volume 1](https://nhsrcindia.org/sites/default/files/Volume%201_SDH-DH_0.pdf)          |
| Vol 2  | CHC + Urban CHC       | [Volume 2](https://nhsrcindia.org/sites/default/files/CHC%20IPHS%202022%20Guidelines%)  |
| Vol 3  | PHC + Urban PHC       | [Volume 3](https://nhsrcindia.org/sites/default/files/PHC%20IPHS_2022_Guideline_pdf.p) |
| Vol 4  | SHC + HWC             | [Volume 4](https://nhsrcindia.org/sites/default/files/SHC-HWC%20%26%20UHWC%20IPHS%202) |

- **Key annexures:** Annexure 2 (services matrix), Annexure 5 (HRH staffing counts), Annexure 7 (diagnostics + equipment list), Annexure 8 (equipment & consumables), bed norms table (p.21)

**15. CPWD Schedule of Rates**  
- **URL:** [CPWD Publications](https://cpwd.gov.in/Publication/Publication.aspx) — also mirrored on state PWD sites; search "CPWD Schedule of Rates 2023" site:gov.in  
- **Auth:** None  
- **Format:** PDF → extract (civil + electrical chapters most relevant for hospital construction)  

**16. National Health Accounts**  
- **Best machine-readable path:** [WHO GHED](https://apps.who.int/nha/database/Select/Indicators/en) — free, no login, CSV export of India NHA data  
- **Direct NHA PDFs:** [NHA Records](https://nhsrcindia.org/national-health-accounts-records) (2013–2023, PDF only)  
- **Key GHED columns:** `country`, `year`, `che_gdp`, `gghed_che`, `pvtd_che`, `oop_che` — broken down by financing scheme, provider type, health function  

---

## Ingestion Priority for `workspace.bronze`

| Priority   | Source                        | Why                                         | Method                      |
|------------|-------------------------------|----------------------------------------------|-----------------------------|
| Do first   | WHO GHO OData                 | No auth, direct JSON→Delta                   | REST call in notebook       |
| Do first   | Nikshay POST API              | No auth, district-level TB                   | BeautifulSoup scrape        |
| Do first   | WHO GHED (NHA)                | No auth, CSV download                        | Direct HTTP                 |
| Do next    | IHME GBD (one account)        | State-level DALYs — core of life_criticality_score | GHDx login + CSV            |
| Do next    | NFHS microdata                | District maternal/neonatal                   | dhsprogram.com registration |
| Do next    | HMIS via data.gov.in          | Facility-level utilization                   | data.gov.in API key         |
| Do next    | PM-JAY HBP Excel              | Procedure cost anchors                       | NHA SETU or SHA mirror      |
| PDF batch  | SRS, NCRB, MoRTH, NCRP, CGHS, IPHS, CPWD | Static reference data                        | pdfplumber/PyMuPDF pipeline |
| Last       | GeM / CPPP tenders            | Real procurement prices                      | Web scrape                  |